In [1]:
import ee
import geemap
import time
import os
from google.colab import files
import geopandas as gpd
import json
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='gee-tutorial-470209')

In [2]:
# Define the desired directory path
target_directory = '/content/Aoi'
uploaded = files.upload(target_directory)

Saving AOI_Ganges_Padam_17.geojson to /content/Aoi/AOI_Ganges_Padam_17.geojson


In [3]:
# Load GeoJSON
with open("/content/Aoi/AOI_Ganges_Padam_17.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)
first_feature = roi.first()
print('Segment Processing Start',first_feature.get('FID').getInfo())

Segment Processing Start 17


# GPT sugested fucode


In [5]:
from tqdm import tqdm
# =================================================
# SCALING FUNCTIONS
# =================================================

def scale_landsat5(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat7(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat8(img):
    sr = img.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

# =================================================
# ACTIVE CHANNEL FUNCTION (CORRECTED)
# =================================================

def get_annual_active_channel(year, satellite):

    start = f"{year-1}-12-01"
    end   = f"{year}-04-01"

    # -------------------------
    # Dataset selection
    # -------------------------
    if satellite == 'LS5':
        collection = "LANDSAT/LT05/C02/T1_L2"
        scale_func = scale_landsat5
        green = 'SR_B2'
        red   = 'SR_B3'
        nir   = 'SR_B4'
        swir1 = 'SR_B5'

    elif satellite == 'LS7':
        collection = "LANDSAT/LE07/C02/T1_L2"
        scale_func = scale_landsat7
        green = 'SR_B2'
        red   = 'SR_B3'
        nir   = 'SR_B4'
        swir1 = 'SR_B5'

    elif satellite == 'LS8':
        collection = "LANDSAT/LC08/C02/T1_L2"
        scale_func = scale_landsat8
        green = 'SR_B3'
        red   = 'SR_B4'
        nir   = 'SR_B5'
        swir1 = 'SR_B6'

    else:
        return None

    col = (ee.ImageCollection(collection)
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', 20))
           .map(scale_func))

    count = col.size().getInfo()
    if count == 0:
        return None

    median = col.median().clip(roi)

    # -------------------------
    # Spectral indices
    # -------------------------
    mndwi = median.normalizedDifference([green, swir1])
    ndvi  = median.normalizedDifference([nir, red])

    mndwi = mndwi.rename('mndwi')
    ndvi  = ndvi.rename('ndvi')

    # Format Satellite name file name (LS_05, LS_07, LS_08)
    if satellite == 'LS5':
      source_display = 'LS_05'
    elif satellite == 'LS7':
      source_display = 'LS_07'
    elif satellite == 'LS8':
      source_display = 'LS_08'

    # # -------------------------
    # # Metadata
    # # -------------------------

    mndwi = mndwi.set({'source_display': source_display,
                       'year': year,
                       'n_images': count})
    ndvi  = ndvi.set({'source_display': source_display,
                      'year': year,
                      'n_images': count})


    return mndwi

# =================================================
# YEAR LIST
# =================================================

ls5_years = [2008]      #list(range(1988, 2012))
ls7_years = [2008]
# ls8_years = list(range(2014, 2027))

pairs = []
pairs.extend([(y,'LS5') for y in ls5_years])
pairs.extend([(y,'LS7') for y in ls7_years])
# pairs.extend([(y,'LS8') for y in ls8_years])

pairs.sort(key=lambda x: x[0])

# =================================================
# PROCESS LOOP
# =================================================

all_masks = []

print("\nGenerating binary masks...\n")

for year, sat in tqdm(pairs):
    try:
        mask = get_annual_active_channel(year, sat)
        if mask:
            all_masks.append(mask)
        else:
            print(f"{sat} {year}  no images")

    except Exception as e:
        print(year, sat, str(e))
    time.sleep(1)

print("\nTotal masks created:", len(all_masks), 'Segemnt', first_feature.get('FID').getInfo())


Generating binary masks...



100%|██████████| 2/2 [00:02<00:00,  1.22s/it]


Total masks created: 2 Segemnt 17


# Export to Drive

In [6]:
# ============================================
# EXPORT ALL MASKS
# ============================================
print("\n🚀 STARTING EXPORTS TO GOOGLE DRIVE")
print("=" * 70)

folder_name = f"FID_{first_feature.get('FID').getInfo()}_Ganges_Padma_MNDWI2"
print(f"📁 Export folder: {folder_name}")
print()

export_tasks = []
for img in tqdm(all_masks, desc="Starting exports", unit="file"):
    year = img.get('year').getInfo()
    source_display = img.get('source_display').getInfo()  # This gives "LS_05", "LS_07", or "LS_08"
    # Create filename exactly as requested: FID_08_Year_LS_05_Active_Channel_Mask
    filename = f"FID_{first_feature.get('FID').getInfo()}_{year}_{source_display}_MNDWI"
    # Get area for display
    try:
        area = img.get('active_area_km2').getInfo()
        area_str = f" ({area:.2f} km²)"
    except:
        area_str = ""
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=filename,
        folder=folder_name,
        fileNamePrefix=filename,
        region=roi.geometry().bounds().getInfo()["coordinates"],
        scale=30,
        crs='EPSG:32645',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    export_tasks.append({'task': task, 'year': year, 'source': source_display, 'filename': filename})
    tqdm.write(f"  🚀 {filename}: Export started{area_str}")
    time.sleep(5)
print(f"\n✅ Started {len(export_tasks)} export tasks to folder: {folder_name}")


🚀 STARTING EXPORTS TO GOOGLE DRIVE
📁 Export folder: FID_17_Ganges_Padma_MNDWI2



Starting exports:   0%|          | 0/2 [00:01<?, ?file/s]

  🚀 FID_17_2008_LS_05_MNDWI: Export started


Starting exports:  50%|█████     | 1/2 [00:07<00:06,  6.28s/file]

  🚀 FID_17_2008_LS_07_MNDWI: Export started


Starting exports: 100%|██████████| 2/2 [00:12<00:00,  6.30s/file]


✅ Started 2 export tasks to folder: FID_17_Ganges_Padma_MNDWI2
